# CS461 Check-in — Automated Produce Quality Inspector
**Baseline Custom CNN + Optimizer comparison (Adam vs SGD+Momentum)**

รันใน **Google Colab (Runtime → T4 GPU)** หรือ **Kaggle Notebook (GPU + Add Data: sriramr/fruits-fresh-and-rotten-for-classification)**
กด *Run all* ครั้งเดียว ทุกกราฟ/ตารางจะถูกบันทึกไว้ใน `outputs/` (บน Colab = Google Drive → `MyDrive/cs461_checkin/outputs`)

**ถ้าหลุดกลางทาง:** Run all ใหม่ได้เลย ส่วนที่เสร็จแล้ว (hash, optimizer ที่เทรนจบ) จะถูกข้าม

ใช้เวลาประมาณ 30–45 นาทีบน T4 · รันบน Mac (Apple Silicon) ได้ด้วย · ไม่มี GPU จะลดเป็นภาพ 128px อัตโนมัติ

In [ ]:
!pip -q install imagehash kagglehub

## 0. Config

In [ ]:
import os, random, time, json, math
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import imagehash
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

CFG = dict(
    data_root=None,      # None = หาให้อัตโนมัติ / หรือใส่ path ที่มีโฟลเดอร์ train/ และ test/
    img_size=224, cache_size=256,
    batch_size=32, epochs=20, patience=5, seed=42,
    weight_decay=1e-4, dropout=0.4,
    dedup_hamming=0,     # 0 = fingerprint ต้องตรงกันเป๊ะ (ทดสอบแล้ว: ไม่มีกลุ่มที่ปนหลายคลาส)
    val_folds=7,         # 1/7 ≈ 14% ของ train เป็น validation
    num_workers=2,
    quick=False,         # True = smoke test (ภาพน้อย, 2 epochs) ไว้เช็กว่าโค้ดรันได้
)
CLASSES = ['freshapples','freshbanana','freshoranges','rottenapples','rottenbanana','rottenoranges']
ROTTEN = {3, 4, 5}
if Path('/var/colab/hostname').exists():      # Colab จริง: เก็บผลลง Google Drive กันหายตอนหลุด
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = Path('/content/drive/MyDrive/cs461_checkin')
elif Path('/kaggle/working').exists():         # Kaggle: ไฟล์ใน /kaggle/working จะโผล่ในแท็บ Output
    BASE = Path('/kaggle/working')
else:                                          # Mac / เครื่องตัวเอง
    BASE = Path('.')
OUT = BASE / 'outputs'; OUT.mkdir(parents=True, exist_ok=True)

DEVICE = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available()   # Apple Silicon Mac (M1/M2/M3/M4)
          else 'cpu')
if DEVICE != 'cuda':
    CFG['num_workers'] = 0          # Mac/CPU ใน notebook: worker process มักพัง
if DEVICE == 'cpu':                 # ไม่มี GPU: ลดภาพเหลือ 128px + epochs น้อยลง (เร็วขึ้น ~3 เท่า)
    CFG.update(img_size=128, cache_size=144, epochs=12, patience=4)
    print('⚠️ No GPU: using 128px images and max 12 epochs. Mention this on your slides.')
AMP = DEVICE == 'cuda'
CACHE = Path(f"/tmp/fruit_cache_{CFG['cache_size']}"); CACHE.mkdir(exist_ok=True)

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.benchmark = True
seed_everything(CFG['seed'])
print('device:', DEVICE)

## 1. Load dataset

In [ ]:
def find_root(starts):
    for d in starts:
        d = Path(d)
        if not d.exists(): continue
        for p in d.rglob('train'):
            if p.is_dir() and (p.parent/'test').is_dir() and all((p/c).is_dir() for c in CLASSES):
                return p.parent
    return None

root = Path(CFG['data_root']) if CFG['data_root'] else find_root(['/kaggle/input', '.'])
if root is None:
    import kagglehub   # ดาวน์โหลดจาก Kaggle (ถ้าโดนถาม auth ให้ตั้ง kaggle.json ตาม README ของ kagglehub)
    root = find_root([kagglehub.dataset_download('sriramr/fruits-fresh-and-rotten-for-classification')])
print('data root:', root)

def list_images(split):
    rows = []
    for i, c in enumerate(CLASSES):
        for f in sorted((root/split/c).iterdir()):
            if f.suffix.lower() in {'.png', '.jpg', '.jpeg'}:
                rows.append((str(f), i, split))
    return pd.DataFrame(rows, columns=['path', 'label', 'split'])

df = pd.concat([list_images('train'), list_images('test')], ignore_index=True)
if CFG['quick']:
    df = df.groupby(['split', 'label']).sample(frac=1, random_state=0).groupby(['split', 'label']).head(40).reset_index(drop=True)
    CFG['epochs'] = 2
print(len(df), 'images')

## 2. EDA

In [ ]:
counts = df.pivot_table(index='label', columns='split', values='path', aggfunc='count')
counts.index = CLASSES
display(counts)
ax = counts.plot.bar(figsize=(9, 4), rot=20, title='Images per class')
ax.set_ylabel('images'); plt.tight_layout(); plt.savefig(OUT/'01_class_counts.png', dpi=150); plt.show()

fig, axes = plt.subplots(len(CLASSES), 5, figsize=(11, 13))
for r, c in enumerate(CLASSES):
    sample = df[(df.label == r) & (df.split == 'train')].sample(5, random_state=1)
    for k, p in enumerate(sample.path):
        axes[r, k].imshow(Image.open(p).convert('RGB')); axes[r, k].axis('off')
    axes[r, 0].set_title(c, loc='left', fontsize=10)
plt.tight_layout(); plt.savefig(OUT/'02_samples.png', dpi=120); plt.show()

sizes = np.array([Image.open(p).size for p in df.path.sample(min(500, len(df)), random_state=0)])
print('image size (w,h) min', sizes.min(0), 'median', np.median(sizes, 0), 'max', sizes.max(0))

## 3. Resize cache + duplicate detection (perceptual hash)
ทุกภาพถูก resize เก็บไว้ครั้งเดียว (เทรนเร็วขึ้นมาก) และคำนวณ pHash แบบ **ทนต่อการหมุน 90°/พลิก**
(เอาค่าต่ำสุดจาก 8 แบบ) แล้วจับกลุ่มภาพที่ต่างกัน ≤ `dedup_hamming` bit เป็น group เดียวกัน

In [ ]:
DIHEDRAL = [None, Image.Transpose.ROTATE_90, Image.Transpose.ROTATE_180, Image.Transpose.ROTATE_270]

HASH_FILE = OUT / f"hashes_v2_{'quick' if CFG['quick'] else 'full'}_{CFG['cache_size']}.npy"
have_hashes = HASH_FILE.exists() and len(np.load(HASH_FILE)) == len(df)

def phash_invariant(im):
    # ตัดเฉพาะกลาง 60% ของภาพ: ขอบดำจากการหมุนในชุดข้อมูลทำให้ภาพต่างผลไม้ได้ hash ใกล้กัน
    g, hs = im.convert('L').crop((int(CFG['cache_size']*.2), int(CFG['cache_size']*.2), int(CFG['cache_size']*.8), int(CFG['cache_size']*.8))), []
    for t in DIHEDRAL:
        a = g if t is None else g.transpose(t)
        for b in (a, a.transpose(Image.Transpose.FLIP_LEFT_RIGHT)):
            hs.append(int(str(imagehash.phash(b)), 16))
    return min(hs)

t0, cache_paths, hs = time.time(), [], []
for i, p in enumerate(df.path):
    out = CACHE / f'{i}.jpg'
    if not out.exists():
        Image.open(p).convert('RGB').resize((CFG['cache_size'],) * 2, Image.BILINEAR).save(out, quality=95)
    cache_paths.append(str(out))
    if not have_hashes:
        hs.append(phash_invariant(Image.open(out)))
    if i % 2000 == 0: print(f'{i}/{len(df)} ({time.time()-t0:.0f}s)')
df['cache'] = cache_paths
if have_hashes:
    hashes = np.load(HASH_FILE); print('loaded saved hashes')
else:
    hashes = np.array(hs, dtype=np.uint64); np.save(HASH_FILE, hashes)
print(f'step done in {time.time()-t0:.0f}s')

POP = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
parent = np.arange(len(hashes))
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a
for i in range(0, len(hashes), 512):
    x = np.ascontiguousarray(hashes[i:i+512, None] ^ hashes[None, :])
    d = POP[x.view(np.uint8)].reshape(x.shape[0], -1, 8).sum(-1)
    for a, b in zip(*np.nonzero(d <= CFG['dedup_hamming'])):
        a += i
        if a < b:
            ra, rb = find(a), find(b)
            if ra != rb: parent[rb] = ra
df['group'] = [find(i) for i in range(len(df))]

gsize = df.group.map(df.group.value_counts())
train_groups = set(df.loc[df.split == 'train', 'group'])
df['leak'] = (df.split == 'test') & df.group.isin(train_groups)
dup_stats = {
    'images_in_duplicate_groups': int((gsize > 1).sum()),
    'pct_in_duplicate_groups': round(float((gsize > 1).mean() * 100), 1),
    'unique_groups': int(df.group.nunique()),
    'test_images_with_near_duplicate_in_train': int(df.leak.sum()),
    'pct_test_leaked': round(float(df.leak.sum() / (df.split == 'test').sum() * 100), 1),
}
n_classes = df.groupby('group').label.nunique()
dup_stats['images_in_mixed_class_groups'] = int(df.group.isin(n_classes[n_classes > 1].index).sum())  # ควรใกล้ 0: ถ้าสูง = จับกลุ่มผิด
dup_stats['largest_group'] = int(gsize.max())
print(json.dumps(dup_stats, indent=2))

# ตัวอย่างกลุ่มภาพซ้ำ ไว้ใส่สไลด์ "Challenges"
big = df[gsize > 1].group.value_counts().head(4).index
fig, axes = plt.subplots(len(big), 5, figsize=(11, 2.4 * max(len(big), 1)), squeeze=False)
for r, g in enumerate(big):
    for k, (_, row) in enumerate(df[df.group == g].head(5).iterrows()):
        axes[r, k].imshow(Image.open(row.path).convert('RGB')); axes[r, k].set_title(row.split, fontsize=8)
    for k in range(5): axes[r, k].axis('off')
plt.suptitle('Near-duplicate groups (rotated/flipped copies)'); plt.tight_layout()
plt.savefig(OUT/'03_duplicates.png', dpi=120); plt.show()

## 4. Split train / val (stratified + group-aware) และ DataLoader

In [ ]:
trn_all = df[df.split == 'train'].reset_index(drop=True)
sgkf = StratifiedGroupKFold(n_splits=CFG['val_folds'], shuffle=True, random_state=CFG['seed'])
tr_idx, va_idx = next(sgkf.split(trn_all, trn_all.label, trn_all.group))
train_df, val_df = trn_all.iloc[tr_idx], trn_all.iloc[va_idx]
test_df = df[df.split == 'test'].reset_index(drop=True)
clean_test_df = test_df[~test_df.leak]
assert not set(train_df.group) & set(val_df.group)
print(f'train {len(train_df)} | val {len(val_df)} | test {len(test_df)} | clean test {len(clean_test_df)}')

MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
S = CFG['img_size']
tf_train = transforms.Compose([
    transforms.RandomResizedCrop(S, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.15),  # ไม่ jitter hue: สีคือสัญญาณความเน่า
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
tf_eval = transforms.Compose([transforms.Resize((S, S)), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class FruitDS(Dataset):
    def __init__(self, frame, tf): self.p, self.y, self.tf = frame.cache.tolist(), frame.label.tolist(), tf
    def __len__(self): return len(self.p)
    def __getitem__(self, i): return self.tf(Image.open(self.p[i]).convert('RGB')), self.y[i]

def loader(frame, tf, shuffle):
    g = torch.Generator().manual_seed(CFG['seed'])
    return DataLoader(FruitDS(frame, tf), batch_size=CFG['batch_size'], shuffle=shuffle, generator=g,
                      num_workers=CFG['num_workers'], pin_memory=DEVICE == 'cuda')
dl_train, dl_val = loader(train_df, tf_train, True), loader(val_df, tf_eval, False)
dl_test, dl_clean = loader(test_df, tf_eval, False), loader(clean_test_df, tf_eval, False)

## 5. Custom CNN (ตาม proposal)
4 × [Conv2D → BatchNorm → ReLU → MaxPool] → Global Avg Pool → FC + Dropout → 6 classes

In [ ]:
class FruitCNN(nn.Module):
    def __init__(self, n_classes=6, chans=(32, 64, 128, 256), p=0.4):
        super().__init__()
        layers, c_in = [], 3
        for c in chans:
            layers += [nn.Conv2d(c_in, c, 3, padding=1, bias=False), nn.BatchNorm2d(c),
                       nn.ReLU(inplace=True), nn.MaxPool2d(2)]
            c_in = c
        self.features = nn.Sequential(*layers)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(p),
                                  nn.Linear(c_in, 128), nn.ReLU(inplace=True), nn.Dropout(p),
                                  nn.Linear(128, n_classes))
    def forward(self, x): return self.head(self.features(x))

n_params = sum(p.numel() for p in FruitCNN().parameters())
print(FruitCNN()); print(f'parameters: {n_params:,}')

## 6. Training loop (early stopping บน val loss)

In [ ]:
@torch.no_grad()
def predict(model, dl):
    model.eval(); ys, ps, losses = [], [], []
    for x, y in dl:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
        with torch.autocast('cuda' if AMP else 'cpu', enabled=AMP):
            out = model(x)
        losses.append(nn.functional.cross_entropy(out.float(), y, reduction='sum').item())
        ys.append(y.cpu()); ps.append(out.float().softmax(1).cpu())
    y, p = torch.cat(ys).numpy(), torch.cat(ps).numpy()
    return y, p, sum(losses) / len(y)

def metrics(y, p):
    pred = p.argmax(1)
    yr, pr = np.isin(y, list(ROTTEN)), np.isin(pred, list(ROTTEN))
    return dict(acc=accuracy_score(y, pred), macro_f1=f1_score(y, pred, average='macro'),
                rotten_recall=recall_score(yr, pr),               # rotten ที่จับได้ (เป้า ≥ 0.95)
                false_reject_rate=float((pr & ~yr).sum() / max((~yr).sum(), 1)))  # fresh ที่โดน reject ผิด

def train(name, make_opt, make_sched=None):
    seed_everything(CFG['seed'])
    model = FruitCNN(p=CFG['dropout']).to(DEVICE)
    opt = make_opt(model.parameters()); sched = make_sched(opt) if make_sched else None
    scaler = torch.amp.GradScaler(enabled=AMP)
    hist, best, best_state, wait = [], math.inf, None, 0
    t0 = time.time()
    for ep in range(1, CFG['epochs'] + 1):
        model.train(); tl, tc, n = 0., 0, 0
        for x, y in dl_train:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            with torch.autocast('cuda' if AMP else 'cpu', enabled=AMP):
                out = model(x); loss = nn.functional.cross_entropy(out, y)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tl += loss.item() * len(y); tc += (out.argmax(1) == y).sum().item(); n += len(y)
        if sched: sched.step()
        yv, pv, vl = predict(model, dl_val); m = metrics(yv, pv)
        hist.append(dict(epoch=ep, train_loss=tl/n, train_acc=tc/n, val_loss=vl, val_acc=m['acc'],
                         val_f1=m['macro_f1'], lr=opt.param_groups[0]['lr']))
        pd.DataFrame(hist).to_csv(OUT/f'{name}_history.csv', index=False)
        print(f"[{name}] ep{ep:02d} train {tl/n:.3f}/{tc/n:.3f} | val {vl:.3f}/{m['acc']:.3f} F1 {m['macro_f1']:.3f}")
        if vl < best - 1e-4:
            best, wait = vl, 0; best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= CFG['patience']: print('early stop'); break
    model.load_state_dict(best_state)
    torch.save(best_state, OUT/f'{name}.pt')
    secs = time.time() - t0
    (OUT/f'{name}_done.json').write_text(json.dumps({'secs': secs}))
    return model, pd.DataFrame(hist), secs

## 7. Experiments: Adam vs SGD + Momentum (setup เดียวกัน, fixed LR)

In [ ]:
EXPERIMENTS = {
    'adam_lr1e-3': lambda ps: torch.optim.Adam(ps, lr=1e-3, weight_decay=CFG['weight_decay']),
    'sgdm_lr1e-2': lambda ps: torch.optim.SGD(ps, lr=1e-2, momentum=0.9, weight_decay=CFG['weight_decay']),
}
runs = {}
for name, make_opt in EXPERIMENTS.items():
    done = OUT/f'{name}_done.json'
    if done.exists() and not CFG['quick']:   # รันจบไปแล้วในรอบก่อน → โหลดมาใช้ ไม่ต้องเทรนใหม่
        model = FruitCNN(p=CFG['dropout']).to(DEVICE)
        model.load_state_dict(torch.load(OUT/f'{name}.pt', map_location=DEVICE))
        hist, secs = pd.read_csv(OUT/f'{name}_history.csv'), json.loads(done.read_text())['secs']
        print(f'[{name}] already finished — loaded from {OUT}')
    else:
        model, hist, secs = train(name, make_opt)
    runs[name] = dict(model=model, hist=hist, secs=secs)

## 8. ผลลัพธ์: กราฟ Loss/Accuracy + ตารางเปรียบเทียบ

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for name, r in runs.items():
    h = r['hist']
    ax[0].plot(h.epoch, h.train_loss, '--', label=f'{name} train'); ax[0].plot(h.epoch, h.val_loss, label=f'{name} val')
    ax[1].plot(h.epoch, h.train_acc, '--', label=f'{name} train'); ax[1].plot(h.epoch, h.val_acc, label=f'{name} val')
    ax[2].plot(h.epoch, h.val_f1, label=name)
for a, t in zip(ax, ['Loss', 'Accuracy', 'Val Macro F1']): a.set_title(t); a.set_xlabel('epoch'); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig(OUT/'04_optimizer_curves.png', dpi=150); plt.show()

@torch.no_grad()
def ms_per_image(model, n=100):
    model.eval(); x = torch.randn(1, 3, S, S, device=DEVICE)
    for _ in range(10): model(x)
    if DEVICE == 'cuda': torch.cuda.synchronize()
    if DEVICE == 'mps': torch.mps.synchronize()
    t = time.time()
    for _ in range(n): model(x)
    if DEVICE == 'cuda': torch.cuda.synchronize()
    if DEVICE == 'mps': torch.mps.synchronize()
    return (time.time() - t) / n * 1000

rows, preds = [], {}
for name, r in runs.items():
    y, p, _ = predict(r['model'], dl_test); preds[name] = (y, p)
    yc, pc, _ = predict(r['model'], dl_clean)
    m, mc = metrics(y, p), metrics(yc, pc)
    rows.append({'run': name, 'epochs_run': len(r['hist']), 'train_min': round(r['secs'] / 60, 1),
                 'best_val_loss': round(r['hist'].val_loss.min(), 4),
                 'test_acc': m['acc'], 'test_macroF1': m['macro_f1'], 'test_rotten_recall': m['rotten_recall'],
                 'test_false_reject': m['false_reject_rate'],
                 'clean_test_macroF1': mc['macro_f1'], 'clean_test_rotten_recall': mc['rotten_recall'],
                 'params': n_params, 'ms_per_img': round(ms_per_image(r['model']), 2)})
table = pd.DataFrame(rows).round(4)
table.to_csv(OUT/'05_comparison.csv', index=False)
display(table)

## 9. Confusion matrix + per-class report (โมเดลที่ดีที่สุด)

In [ ]:
best_name = table.sort_values('best_val_loss').iloc[0].run   # เลือกจาก validation ไม่ใช่ test
y, p = preds[best_name]; pred = p.argmax(1)
print('best (by val loss):', best_name)
print(classification_report(y, pred, target_names=CLASSES, digits=3))
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix(y, pred), display_labels=CLASSES).plot(ax=ax, xticks_rotation=35, cmap='Blues', colorbar=False)
ax.set_title(f'Confusion matrix — {best_name} (test)'); plt.tight_layout()
plt.savefig(OUT/'06_confusion_matrix.png', dpi=150); plt.show()

## 10. ภาพที่ทายผิด (ใช้คุยเรื่อง Challenges)

In [ ]:
wrong = np.nonzero(pred != y)[0]
wrong = wrong[np.argsort(-p[wrong].max(1))][:12]   # ผิดแบบมั่นใจสูงสุดก่อน
if len(wrong):
    cols = 4; rows_ = math.ceil(len(wrong) / cols)
    fig, axes = plt.subplots(rows_, cols, figsize=(12, 3.2 * rows_), squeeze=False)
    for a in axes.flat: a.axis('off')
    for a, i in zip(axes.flat, wrong):
        a.imshow(Image.open(test_df.path[i]).convert('RGB'))
        a.set_title(f'true {CLASSES[y[i]]}\npred {CLASSES[pred[i]]} ({p[i].max():.2f})', fontsize=8)
    plt.tight_layout(); plt.savefig(OUT/'07_misclassified.png', dpi=120); plt.show()
else:
    print('no errors on test set')

## 11. บันทึกสรุปสำหรับสไลด์

In [ ]:
summary = dict(config=CFG, device=DEVICE, split_sizes=dict(train=len(train_df), val=len(val_df), test=len(test_df),
               clean_test=len(clean_test_df)), duplicates=dup_stats, best_run=best_name,
               results=table.to_dict(orient='records'))
(OUT/'results.json').write_text(json.dumps(summary, indent=2, default=str))
print(json.dumps(summary, indent=2, default=str))
print('saved to', OUT.resolve())